# Pilot Study: Gemini 3.1 Pro Preview Timestamp Localization

This notebook runs a zero-shot pilot on the generated mixed audio set and asks Gemini to:
- predict one degradation timestamp interval
- provide a one-sentence description of the degradation

Results are printed live file-by-file and summarized at the end.


In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from google import genai
from google.genai import types

In [ ]:
MODEL_NAME = "gemini-3.1-pro-preview"
MAX_FILES = 40
TEMPERATURE = 0
TOP_P = 0.95
THINKING_BUDGET = 256

MANIFEST_PATH = Path("../data/processed/nisqa_sim_mix_lowmos_active_40/manifest.csv")
MIX_DIR = Path("../data/processed/nisqa_sim_mix_lowmos_active_40")
OUTPUT_CSV = Path("../results/analysis/gemini31_pilot_localization_results.csv")

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "Missing API key. Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your notebook environment, "
        "then rerun this cell. Example in a notebook cell: %env GEMINI_API_KEY=your_key_here"
    )

os.environ["GEMINI_API_KEY"] = api_key
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=120000),
)

print(f"Model: {MODEL_NAME}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Mix dir: {MIX_DIR}")
print("API key source: env")


SyntaxError: invalid syntax (3055188729.py, line 11)

In [ ]:
def parse_json_payload(text: str) -> dict | None:
    text = text.strip()
    if not text:
        return None

    fenced = re.search(r"```(?:json)?\s*(\{[\s\S]*?\})\s*```", text)
    candidate = fenced.group(1) if fenced else None
    if candidate is None:
        direct = re.search(r"(\{[\s\S]*\})", text)
        candidate = direct.group(1) if direct else None

    if candidate is None:
        return None

    try:
        return json.loads(candidate)
    except Exception:
        return None


def parse_first_interval(mix_deg_segments_json: str) -> tuple[float, float]:
    items = json.loads(mix_deg_segments_json)
    first = items[0]
    return float(first["start"]), float(first["end"])


def clamp_interval(start: float, end: float, duration: float) -> tuple[float, float]:
    start = max(0.0, min(float(start), duration))
    end = max(0.0, min(float(end), duration))
    if end < start:
        start, end = end, start
    if end == start:
        end = min(duration, start + 0.05)
    return start, end


def interval_iou(a0: float, a1: float, b0: float, b1: float) -> float:
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    if union <= 0:
        return 0.0
    return inter / union


def build_mix_path(
    index_value: int,
    filename_deg: str,
    mix_filename: str | None = None,
    index_width: int = 3,
) -> Path:
    if mix_filename is not None:
        name = str(mix_filename).strip()
        if name and name.lower() not in {"nan", "none"}:
            return MIX_DIR / name

    stem = Path(filename_deg).stem
    return MIX_DIR / f"{index_value:0{index_width}d}_mix_{stem}.wav"


In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH).sort_values("index").reset_index(drop=True)
if MAX_FILES is not None:
    manifest_df = manifest_df.head(MAX_FILES).copy()

max_index = int(manifest_df["index"].max()) if len(manifest_df) else 0
index_width = max(3, len(str(max_index)))

manifest_df["mix_path"] = manifest_df.apply(
    lambda row: str(
        build_mix_path(
            int(row["index"]),
            row["filename_deg"],
            row.get("mix_filename", None),
            index_width,
        )
    ),
    axis=1,
)

missing = [p for p in manifest_df["mix_path"].tolist() if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} mix files. First missing: {missing[0]}")

display(manifest_df[["index", "filename_deg", "mos", "mix_deg_segments"]].head(10))
print(f"Files queued for pilot: {len(manifest_df)}")

,index,filename_deg,mos,mix_deg_segments
0,0,c09587_3_235_2_7_001-ch6-speaker_seg101.wav,1.8,"[{""start"": 6.567, ""end"": 7.792}]"
1,1,c01477_4_958_2_7_001-ch6-speaker_seg98.wav,1.8,"[{""start"": 3.156, ""end"": 5.796}]"
2,2,c00442_som_03502_02012401285_seg.wav,2.2,"[{""start"": 7.096, ""end"": 8.14}]"
3,3,c05041_book_04474_chp_0040_reader_10730_1_seg.wav,1.8,"[{""start"": 0.697, ""end"": 1.862}]"
4,4,c04335_book_00839_chp_0013_reader_10759_4_seg.wav,1.8,"[{""start"": 4.041, ""end"": 5.979}]"
5,5,c03722_4_148_2_7_001-ch6-speaker_seg55.wav,1.6,"[{""start"": 3.506, ""end"": 4.712}]"
6,6,c01865_1_965_2_7_001-ch6-speaker_seg9.wav,2.0,"[{""start"": 2.72, ""end"": 4.906}]"
7,7,c01344_3_726_2_7_001-ch6-speaker_seg46.wav,1.2,"[{""start"": 0.0, ""end"": 3.3}]"
8,8,c09808_4_448_2_7_001-ch6-speaker_seg92.wav,1.6,"[{""start"": 3.278, ""end"": 5.738}]"
9,9,c08926_mim_03397_00063835793_seg.wav,1.0,"[{""start"": 1.178, ""end"": 2.728}]"


Files queued for pilot: 40


In [ ]:
PROMPT_TEMPLATE = """
You are localizing ONE injected degradation interval in a speech clip.
Clip duration: {duration_seconds:.2f} seconds.

Return exactly one best-guess interval where degradation is strongest and most localized.
Do NOT default to the full clip unless the entire clip is clearly degraded.
Use decimal seconds with at least 2 decimal places for both times (example: 1.37, 2.84).

Output strictly as JSON with keys:
{{
  "start_time_sec": <float>,
  "end_time_sec": <float>,
  "description": "<short phrase, max 12 words, artifact-specific>"
}}
Do not output anything outside JSON.
"""

results = []
total = len(manifest_df)

for i, row in manifest_df.iterrows():
    idx = int(row["index"])
    mix_path = Path(row["mix_path"])
    duration = float(row["duration_seconds"])
    gt_start, gt_end = parse_first_interval(row["mix_deg_segments"])

    with open(mix_path, "rb") as f:
        audio_bytes = f.read()

    prompt_text = PROMPT_TEMPLATE.format(duration_seconds=duration)

    print(
        f"[{i+1:02d}/{total:02d}] idx={idx:03d} | querying Gemini...",
        flush=True,
    )

    response_text = ""
    description = ""

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[
                types.Part.from_bytes(data=audio_bytes, mime_type="audio/wav"),
                prompt_text,
            ],
            config=types.GenerateContentConfig(
                temperature=TEMPERATURE,
                top_p=TOP_P,
                response_mime_type="application/json",
                thinking_config=types.ThinkingConfig(
                    thinking_budget=THINKING_BUDGET,
                ),
            ),
        )
        response_text = response.text or ""
    except Exception as exc:
        response_text = f"API_ERROR: {exc}"

    payload = parse_json_payload(response_text)

    if payload is None:
        pred_start, pred_end = 0.0, min(duration, 0.1)
        description = "Failed to parse model JSON output."
    else:
        pred_start = float(payload.get("start_time_sec", 0.0))
        pred_end = float(payload.get("end_time_sec", pred_start + 0.1))
        description = str(payload.get("description", "")).strip()

    pred_start, pred_end = round(pred_start, 2), round(pred_end, 2)
    pred_start, pred_end = clamp_interval(pred_start, pred_end, duration)

    iou = interval_iou(gt_start, gt_end, pred_start, pred_end)
    start_abs_err = abs(pred_start - gt_start)
    end_abs_err = abs(pred_end - gt_end)

    rec = {
        "index": idx,
        "filename_deg": row["filename_deg"],
        "mix_filename": str(row.get("mix_filename", "")),
        "model_used": MODEL_NAME,
        "mos": float(row["mos"]),
        "duration_seconds": duration,
        "gt_start": gt_start,
        "gt_end": gt_end,
        "pred_start": pred_start,
        "pred_end": pred_end,
        "iou": iou,
        "start_abs_err": start_abs_err,
        "end_abs_err": end_abs_err,
        "description": description,
        "raw_response": response_text,
    }
    results.append(rec)

    print(
        f"[{i+1:02d}/{total:02d}] idx={idx:03d} | pred=({pred_start:.2f}, {pred_end:.2f}) "
        f"| gt=({gt_start:.2f}, {gt_end:.2f}) | IoU={iou:.3f} | {description}",
        flush=True,
    )

    if (i + 1) % 5 == 0 or (i + 1) == total:
        running_df = pd.DataFrame(results)
        display(running_df.tail(min(5, len(running_df))))


[01/40] idx=000 | querying Gemini...
[01/40] idx=000 | pred=(0.00, 8.00) | gt=(6.57, 7.79) | IoU=0.153 | No clear localized degradation found, assuming full clip.
[02/40] idx=001 | querying Gemini...


In [ ]:
results_df = pd.DataFrame(results).sort_values("index").reset_index(drop=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")

summary = {
    "n_files": len(results_df),
    "mean_iou": float(results_df["iou"].mean()),
    "median_iou": float(results_df["iou"].median()),
    "hit_iou_ge_0.1": float((results_df["iou"] >= 0.1).mean()),
    "hit_iou_ge_0.3": float((results_df["iou"] >= 0.3).mean()),
    "mean_start_abs_err": float(results_df["start_abs_err"].mean()),
    "mean_end_abs_err": float(results_df["end_abs_err"].mean()),
}

display(pd.DataFrame([summary]))
display(results_df[[
    "index",
    "filename_deg",
    "gt_start",
    "gt_end",
    "pred_start",
    "pred_end",
    "iou",
    "description",
]].head(20))

## Player + Overlay

Use this section to listen to a sample and inspect where the true degradation (`GT`) and model prediction (`Pred`) land on the waveform.


In [ ]:
if len(results_df) == 0:
    raise ValueError("No pilot results found. Run the inference loop cell first.")

viz_df = results_df.copy().sort_values("index").reset_index(drop=True)
max_index = int(viz_df["index"].max()) if len(viz_df) else 0
index_width = max(3, len(str(max_index)))

viz_df["mix_path"] = viz_df.apply(
    lambda row: str(
        build_mix_path(
            int(row["index"]),
            row["filename_deg"],
            row.get("mix_filename", None),
            index_width,
        )
    ),
    axis=1,
)

def _load_audio_mono(path: str) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), int(sr)


def _show_case(row_id: int) -> None:
    row = viz_df.iloc[int(row_id)]
    audio, sr = _load_audio_mono(row["mix_path"])
    times = np.arange(len(audio)) / sr

    gt_start, gt_end = float(row["gt_start"]), float(row["gt_end"])
    pred_start, pred_end = float(row["pred_start"]), float(row["pred_end"])

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(times, audio, color="#111111", linewidth=0.65)
    ax.axvspan(gt_start, gt_end, color="#ffb347", alpha=0.28, label="GT")
    ax.axvspan(pred_start, pred_end, color="#66c2a5", alpha=0.24, label="Pred")

    ax.set_title("Waveform with GT vs Predicted Degradation Interval")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    print(f"idx={int(row['index']):03d} | file={row['filename_deg']}")
    print(f"IoU={row['iou']:.3f} | GT=({gt_start:.2f}, {gt_end:.2f}) | Pred=({pred_start:.2f}, {pred_end:.2f})")
    print(f"Description: {row['description']}")
    display(Audio(filename=row["mix_path"]))


try:
    import ipywidgets as widgets

    options = [
        (
            f"idx={int(r['index']):03d} | IoU={r['iou']:.3f} | {r['filename_deg']}",
            i,
        )
        for i, r in viz_df.iterrows()
    ]

    selector = widgets.Dropdown(
        options=options,
        value=0,
        description="Sample",
        layout=widgets.Layout(width="95%"),
    )

    out = widgets.interactive_output(_show_case, {"row_id": selector})
    display(selector, out)
except Exception:
    print("ipywidgets not available, showing first 5 samples.")
    for rid in range(min(5, len(viz_df))):
        _show_case(rid)
